# Bahdanau attention against Luong attention
The three score functions by hand on small numbers, then the same encoder–decoder trained with Bahdanau's
additive attention and with Luong's dot and general attention: parameters, time, BLEU and attention weights.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"          # hide TensorFlow's start-up messages
import re, time, random, collections
import numpy as np
import pandas as pd
import keras
import tensorflow as tf

np.set_printoptions(precision=4, suppress=True)
print(tf.config.list_physical_devices("GPU"))

I0000 00:00:1791079678.764604 1750914 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Scores by hand: four encoder states and one decoder state, all of size 4

In [2]:
rng = np.random.default_rng(0)
Hs = rng.uniform(-1, 1, (4, 4)).round(1)          # h_1..h_4, one per row
s = np.array([0.5, -0.2, 0.8, 0.1])               # decoder state
print("encoder states:\n", Hs, "\ndecoder state:", s)
softmax = lambda z: np.exp(z - z.max()) / np.exp(z - z.max()).sum()

encoder states:
 [[ 0.3 -0.5 -0.9 -1. ]
 [ 0.6  0.8  0.2  0.5]
 [ 0.1  0.9  0.6 -1. ]
 [ 0.7 -0.9  0.5 -0.6]] 
decoder state: [ 0.5 -0.2  0.8  0.1]


**Bahdanau (additive, concat).** Stack [s ; h_j] for every j: a 4 x 8 matrix. A hidden layer of 3 tanh units
(weights 8 x 3) and an output unit (weights 3 x 1) give one score per row.

In [3]:
W = rng.uniform(-1, 1, (8, 3)).round(1)
v = rng.uniform(-1, 1, (3,)).round(1)
X = np.hstack([np.tile(s, (4, 1)), Hs])          # 4 x 8
hidden = np.tanh(X @ W)                           # 4 x 3
e_add = hidden @ v                                # 4 scores
print("concatenated input:\n", X)
print("hidden layer:\n", hidden.round(3))
print("scores e:", e_add.round(3), " weights alpha:", softmax(e_add).round(3))
# the same as W_s s + U h_j, with W split into its top and bottom halves
print("split form equal:", np.allclose(np.tanh(s @ W[:4] + Hs @ W[4:]) @ v, e_add))

concatenated input:
 [[ 0.5 -0.2  0.8  0.1  0.3 -0.5 -0.9 -1. ]
 [ 0.5 -0.2  0.8  0.1  0.6  0.8  0.2  0.5]
 [ 0.5 -0.2  0.8  0.1  0.1  0.9  0.6 -1. ]
 [ 0.5 -0.2  0.8  0.1  0.7 -0.9  0.5 -0.6]]
hidden layer:
 [[-0.06   0.438  0.414]
 [ 0.804  0.508  0.558]
 [-0.264 -0.824  0.726]
 [ 0.565  0.585  0.31 ]]
scores e: [-0.099 -0.011  0.449 -0.116]  weights alpha: [0.208 0.227 0.36  0.205]
split form equal: True


**Luong dot** and **Luong general** (a learned 4 x 4 matrix between the two states).

In [4]:
e_dot = Hs @ s
Wa = rng.uniform(-1, 1, (4, 4)).round(1)
e_gen = Hs @ (Wa.T @ s)                           # s^T W_a h_j for every j
for name, e in (("dot", e_dot), ("general", e_gen)):
    print(f"{name:8s} scores {e.round(3)}  weights {softmax(e).round(3)}")
print("parameters of the score:  additive", W.size + v.size, "| dot 0 | general", Wa.size)

dot      scores [-0.57  0.35  0.25  0.87]  weights [0.1   0.251 0.227 0.422]
general  scores [ 0.477 -0.254  0.975 -0.887]  weights [0.296 0.142 0.486 0.076]
parameters of the score:  additive 27 | dot 0 | general 16


## The data
English–French pairs (the Keras example dataset), grouped by their English side so that test sentences
never appear in training. English sentences of up to 16 words. Long sentences are rare, so the training
set keeps every pair with 9 or more English words and fills up to 60,000 pairs with shorter ones.

In [5]:
path = keras.utils.get_file("fra-eng.zip", origin="http://storage.googleapis.com/download.tensorflow.org/data/fra-eng.zip",
                            extract=True)
lines = open(os.path.join(path, "fra.txt"), encoding="utf8").read().strip().split("\n")
tok = lambda s: re.findall(r"[\w']+|[^\w\s]", s.lower())
groups = collections.defaultdict(list)
for line in lines:
    e_, f_ = line.split("\t")[:2]
    e_, f_ = tok(e_), tok(f_)
    if len(e_) <= 16 and len(f_) <= 20:
        groups[tuple(e_)].append(f_)
BUCKETS = [(1, 4), (5, 7), (8, 10), (11, 13), (14, 16)]
bucket = lambda n: next(i for i, (a, b) in enumerate(BUCKETS) if a <= n <= b)
keys = sorted(groups)
random.Random(0).shuffle(keys)
test_keys, rest = [], []
taken = collections.Counter()
for k in keys:
    b = bucket(len(k))
    if taken[b] < 200:
        test_keys.append(k); taken[b] += 1
    else:
        rest.append(k)
pairs = [(list(k), f) for k in rest for f in groups[k]]
long_ = [p for p in pairs if len(p[0]) >= 9]
short = [p for p in pairs if len(p[0]) < 9]
random.Random(1).shuffle(short)
train = long_ + short[:60000 - len(long_)]
random.Random(2).shuffle(train)
print(len(train), "training pairs,", len(long_), "of them with 9+ English words;", len(test_keys), "test sentences",
      dict(sorted(taken.items())))

60000 training pairs, 39070 of them with 9+ English words; 1000 test sentences {0: 200, 1: 200, 2: 200, 3: 200, 4: 200}


In [6]:
def vocab(seqs, n):
    cnt = collections.Counter(w for s in seqs for w in s)
    words = ["<pad>", "<unk>", "<start>", "<end>"] + [w for w, _ in cnt.most_common(n)]
    return {w: i for i, w in enumerate(words)}, words

en_ix, en_w = vocab([e_ for e_, _ in train], 8000)
fr_ix, fr_w = vocab([f_ for _, f_ in train], 10000)
LE, LF = 16, 22

def encode(seqs, ix, L, add=False):
    X = np.zeros((len(seqs), L), "int32")
    for i, s in enumerate(seqs):
        s = ["<start>"] + s + ["<end>"] if add else s
        ids = [ix.get(w, 1) for w in s][:L]
        X[i, :len(ids)] = ids
    return X

Xe = encode([e_ for e_, _ in train], en_ix, LE)
Yf = encode([f_ for _, f_ in train], fr_ix, LF, add=True)
Xd, Yd = Yf[:, :-1], Yf[:, 1:]
test_en = [list(k) for k in test_keys]
test_refs = [groups[k] for k in test_keys]
Xe_test = encode(test_en, en_ix, LE)
print(len(en_w), len(fr_w), Xe.shape, Xd.shape)

8004 10004 (60000, 16) (60000, 21)


## The two models, written as one loop over the decoder steps
Encoder: embedding + bidirectional LSTM (128 units each way, as in Bahdanau et al. 2015, section 3.2), keeping
every hidden state h_1..h_n (forward and backward joined: 256 numbers). Decoder: an LSTM cell, one step per French
word. Without attention the decoder starts from the encoder's final states and sees nothing else.
With attention (Bahdanau et al. 2015), at step i:
e_ij = v^T tanh(W s_{i-1} + U h_j), alpha_i = softmax over j (padding excluded), c_i = sum_j alpha_ij h_j,
and c_i joins the previous word as the LSTM cell's input and the new state as the softmax layer's input.

In [7]:
D, H, A = 128, 256, 128

class Seq2Seq(keras.Model):
    def __init__(self, attention, seed):
        super().__init__()
        keras.utils.set_random_seed(seed)
        self.attention = attention
        self.emb_e = keras.layers.Embedding(len(en_w), D)
        self.enc = keras.layers.Bidirectional(keras.layers.LSTM(H // 2, return_sequences=True, return_state=True))
        self.emb_d = keras.layers.Embedding(len(fr_w), D)
        self.cell = keras.layers.LSTMCell(H)
        self.out = keras.layers.Dense(len(fr_w))
        if attention:
            self.W = keras.layers.Dense(A, use_bias=False)   # applied to s_{i-1}
            self.U = keras.layers.Dense(A)                   # applied to h_j
            self.v = keras.layers.Dense(1, use_bias=False)

    def call(self, xe, first, gold=None, steps=LF - 1):
        hs, hf, cf, hb, cb = self.enc(self.emb_e(xe))      # forward and backward final states
        h, c = tf.concat([hf, hb], -1), tf.concat([cf, cb], -1)
        pad = tf.cast(xe == 0, tf.float32)                   # 1 where the source is padding
        if self.attention:
            Uh = self.U(hs)                                  # does not depend on i: computed once
        word, logits, alphas = first, [], []
        for t in range(steps):
            x = self.emb_d(word)
            if self.attention:
                e = tf.squeeze(self.v(tf.tanh(self.W(h)[:, None, :] + Uh)), -1) - 1e9 * pad
                a = tf.nn.softmax(e, axis=-1)
                ctx = tf.reduce_sum(a[:, :, None] * hs, axis=1)
                alphas.append(a)
                x = tf.concat([x, ctx], -1)
            o, (h, c) = self.cell(x, [h, c])
            lg = self.out(tf.concat([o, ctx], -1) if self.attention else o)
            logits.append(lg)
            word = gold[:, t + 1] if (gold is not None and t + 1 < steps) else tf.argmax(lg, -1, output_type=tf.int32)
        return tf.stack(logits, 1), (tf.stack(alphas, 1) if alphas else None)

loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction="none")

def ngrams(s, n):
    return collections.Counter(tuple(s[i:i + n]) for i in range(len(s) - n + 1))

def bleu(hyps, refss):
    """Corpus BLEU-4 with several references per sentence (Papineni et al. 2002), 0-100."""
    match, total = np.zeros(4), np.zeros(4)
    hyp_len = ref_len = 0
    for hy, refs in zip(hyps, refss):
        hyp_len += len(hy)
        ref_len += min((abs(len(r) - len(hy)), len(r)) for r in refs)[1]
        for n in range(1, 5):
            best = collections.Counter()
            for r in refs:
                best |= ngrams(r, n)
            match[n - 1] += sum(min(k, best[g]) for g, k in ngrams(hy, n).items())
            total[n - 1] += max(len(hy) - n + 1, 0)
    if match.min() == 0:
        return 0.0
    bp = 1.0 if hyp_len > ref_len else np.exp(1 - ref_len / max(hyp_len, 1))
    return 100 * bp * np.exp(np.mean(np.log(match / total)))

def greedy(m, X, batch=100):
    """Greedy decoding in batches (keeps GPU memory small); returns words and attention weights."""
    ids, als = [], []
    for k in range(0, len(X), batch):
        lg, al = m(X[k:k + batch], tf.fill([len(X[k:k + batch])], fr_ix["<start>"]))
        ids.append(tf.argmax(lg, -1).numpy())
        als.append(al.numpy() if al is not None else None)
    al = np.concatenate(als) if als[0] is not None else None
    out = []
    for row in np.concatenate(ids):
        s = []
        for k in row:
            if fr_w[k] == "<end>":
                break
            s.append(fr_w[k])
        out.append(s)
    return out, al

def train_model(attention, seed, epochs=12):
    m = Seq2Seq(attention, seed)
    opt = keras.optimizers.Adam()
    m(Xe[:2], Xd[:2, 0], Xd[:2])

    @tf.function
    def step(xe, xd, yd):
        with tf.GradientTape() as tape:
            lg, _ = m(xe, xd[:, 0], xd)
            mask = tf.cast(yd > 0, tf.float32)
            loss = tf.reduce_sum(loss_fn(yd, lg) * mask) / tf.reduce_sum(mask)
        opt.apply_gradients(zip(tape.gradient(loss, m.trainable_variables), m.trainable_variables))
        return loss

    ds = tf.data.Dataset.from_tensor_slices((Xe, Xd, Yd)).shuffle(len(Xe), seed=seed).batch(128)
    t0 = time.time()
    for ep in range(epochs):
        loss = np.mean([float(step(a, b, c)) for a, b, c in ds])
    print(f"attention={attention} seed={seed}: final training loss {loss:.3f}, {time.time() - t0:.0f} s,",
          f"{m.count_params():,} parameters")
    return m

## Luong's global attention (Luong et al. 2015, eq. 5-7)
The decoder LSTM runs first; its current state s_i scores every encoder state h_j (dot: s_i . h_j, general:
s_i^T W h_j). The context c_i is joined to s_i *after* the LSTM step: h~_i = tanh(W_c [c_i ; s_i]), and the
softmax layer reads h~_i. Without input feeding the decoder LSTM never needs c_i, so in training it runs over
the whole gold sentence in one call, and attention for all steps is two matrix products.

In [8]:
class Luong(keras.Model):
    def __init__(self, score, seed):
        super().__init__()
        keras.utils.set_random_seed(seed)
        self.score = score
        self.emb_e = keras.layers.Embedding(len(en_w), D)
        self.enc = keras.layers.Bidirectional(keras.layers.LSTM(H // 2, return_sequences=True, return_state=True))
        self.emb_d = keras.layers.Embedding(len(fr_w), D)
        self.dec = keras.layers.LSTM(H, return_sequences=True, return_state=True)
        self.Wa = keras.layers.Dense(H, use_bias=False) if score == "general" else None
        self.Wc = keras.layers.Dense(H, activation="tanh", use_bias=False)
        self.out = keras.layers.Dense(len(fr_w))

    def attend(self, S, hs, pad):
        keys = self.Wa(hs) if self.Wa is not None else hs           # general: s^T W h = s . (W h)
        e = tf.einsum("bid,bjd->bij", S, keys) - 1e9 * pad[:, None, :]
        a = tf.nn.softmax(e, axis=-1)
        ctx = tf.einsum("bij,bjd->bid", a, hs)
        return self.out(self.Wc(tf.concat([ctx, S], -1))), a

    def call(self, xe, first, gold=None, steps=LF - 1):
        hs, hf, cf, hb, cb = self.enc(self.emb_e(xe))      # forward and backward final states
        h, c = tf.concat([hf, hb], -1), tf.concat([cf, cb], -1)
        pad = tf.cast(xe == 0, tf.float32)
        if gold is not None:                                         # training: all steps at once
            S, _, _ = self.dec(self.emb_d(gold[:, :steps]), initial_state=[h, c])
            return self.attend(S, hs, pad)
        word, logits, alphas = first, [], []                         # prediction: one word at a time
        for t in range(steps):
            o, h, c = self.dec(self.emb_d(word[:, None]), initial_state=[h, c])
            lg, a = self.attend(o, hs, pad)
            logits.append(lg[:, 0]); alphas.append(a[:, 0])
            word = tf.argmax(lg[:, 0], -1, output_type=tf.int32)
        return tf.stack(logits, 1), tf.stack(alphas, 1)

def train_any(m, seed, epochs=12):
    opt = keras.optimizers.Adam()
    m(Xe[:2], Xd[:2, 0], Xd[:2])

    @tf.function
    def step(xe, xd, yd):
        with tf.GradientTape() as tape:
            lg, _ = m(xe, xd[:, 0], xd)
            mask = tf.cast(yd > 0, tf.float32)
            loss = tf.reduce_sum(loss_fn(yd, lg) * mask) / tf.reduce_sum(mask)
        opt.apply_gradients(zip(tape.gradient(loss, m.trainable_variables), m.trainable_variables))
        return loss

    ds = tf.data.Dataset.from_tensor_slices((Xe, Xd, Yd)).shuffle(len(Xe), seed=seed).batch(128)
    times = []
    for ep in range(epochs):
        t0 = time.time()
        loss = np.mean([float(step(a, b, c)) for a, b, c in ds])
        times.append(time.time() - t0)
    return loss, float(np.median(times[1:]))       # epoch 1 includes tracing the graph

## Training: Bahdanau (concat), Luong dot, Luong general; 12 epochs, 3 seeds each

In [9]:
rows, models = [], {}
for kind in ("Bahdanau (concat)", "Luong dot", "Luong general"):
    for seed in range(3):
        m = Seq2Seq(True, seed) if kind.startswith("Bahdanau") else Luong(kind.split()[1], seed)
        loss, sec = train_any(m, seed)
        models[(kind, seed)] = m
        hyp, _ = greedy(m, Xe_test)
        long_idx = [i for i, k in enumerate(test_keys) if len(k) >= 11]
        rows.append(dict(model=kind, seed=seed, params=m.count_params(), sec_per_epoch=sec, train_loss=loss,
                         bleu_all=bleu(hyp, test_refs),
                         bleu_11plus=bleu([hyp[i] for i in long_idx], [test_refs[i] for i in long_idx])))
        print(rows[-1])
res = pd.DataFrame(rows)
res.to_csv("data/comparison.csv", index=False)
print(res.groupby("model")[["params", "sec_per_epoch", "train_loss", "bleu_all", "bleu_11plus"]].mean().round(2))

{'model': 'Bahdanau (concat)', 'seed': 0, 'params': 8422420, 'sec_per_epoch': 52.78645324707031, 'train_loss': np.float64(0.752851830997955), 'bleu_all': np.float64(25.246825089229503), 'bleu_11plus': np.float64(20.551335581857565)}


{'model': 'Bahdanau (concat)', 'seed': 1, 'params': 8422420, 'sec_per_epoch': 51.060340881347656, 'train_loss': np.float64(0.796053565641456), 'bleu_all': np.float64(23.84817732163775), 'bleu_11plus': np.float64(18.525146202350168)}


{'model': 'Bahdanau (concat)', 'seed': 2, 'params': 8422420, 'sec_per_epoch': 48.8933424949646, 'train_loss': np.float64(0.752432167402971), 'bleu_all': np.float64(26.33055312972077), 'bleu_11plus': np.float64(20.912729205921725)}


{'model': 'Luong dot', 'seed': 0, 'params': 5664532, 'sec_per_epoch': 31.713643074035645, 'train_loss': np.float64(0.7952329859550573), 'bleu_all': np.float64(31.86894020284074), 'bleu_11plus': np.float64(27.205748845809154)}


{'model': 'Luong dot', 'seed': 1, 'params': 5664532, 'sec_per_epoch': 20.04832148551941, 'train_loss': np.float64(0.791930651232632), 'bleu_all': np.float64(30.8784116417348), 'bleu_11plus': np.float64(26.0907441111206)}


{'model': 'Luong dot', 'seed': 2, 'params': 5664532, 'sec_per_epoch': 13.588922500610352, 'train_loss': np.float64(0.8035673466064274), 'bleu_all': np.float64(32.06115580466824), 'bleu_11plus': np.float64(27.060579024050696)}


{'model': 'Luong general', 'seed': 0, 'params': 5730068, 'sec_per_epoch': 19.81926965713501, 'train_loss': np.float64(1.0622582475005438), 'bleu_all': np.float64(17.713974481026163), 'bleu_11plus': np.float64(12.493670975931254)}


{'model': 'Luong general', 'seed': 1, 'params': 5730068, 'sec_per_epoch': 19.554060697555542, 'train_loss': np.float64(1.0521704051286174), 'bleu_all': np.float64(17.88001674204347), 'bleu_11plus': np.float64(11.904637639542965)}


{'model': 'Luong general', 'seed': 2, 'params': 5730068, 'sec_per_epoch': 19.4268479347229, 'train_loss': np.float64(1.0750634471999048), 'bleu_all': np.float64(17.44303714484776), 'bleu_11plus': np.float64(11.443100685214901)}
                      params  sec_per_epoch  train_loss  bleu_all  bleu_11plus
model                                                                         
Bahdanau (concat)  8422420.0          50.91        0.77     25.14        20.00
Luong dot          5664532.0          21.78        0.80     31.60        26.79
Luong general      5730068.0          19.60        1.06     17.68        11.95


## The attention weights of the three models on one real sentence
Each model translates the same test sentence; the weights are saved for the figure.

In [10]:
cands = [i for i, k in enumerate(test_keys) if 9 <= len(k) <= 12]
hyp_b, _ = greedy(models[("Bahdanau (concat)", 0)], Xe_test[cands])
sc = [bleu([hy], [test_refs[i]]) for hy, i in zip(hyp_b, cands)]
order = np.argsort(sc)[::-1]
for j in order[:6]:
    print(round(sc[j], 1), " ".join(test_en[cands[j]]), "->", " ".join(hyp_b[j]))
pick = cands[int(order[0])]
src = test_en[pick]
out = []
for kind in ("Bahdanau (concat)", "Luong dot", "Luong general"):
    hy, al = greedy(models[(kind, 0)], Xe_test[pick:pick + 1])
    hy = hy[0]
    W_ = al[0][:len(hy) + 1, :len(src)]
    print(kind, ":", " ".join(hy))
    for i, w in enumerate(hy + ["<end>"]):
        for j, x in enumerate(src):
            out.append(dict(model=kind, i=i, j=j, french=w, english=x, alpha=float(W_[i, j])))
pd.DataFrame(out).to_csv("data/attention_three.csv", index=False)

100.0 she advised him to talk about his life in america . -> elle lui a conseillé de parler de sa vie en amérique .
100.0 instead of going to school , he stayed at home . -> au lieu d'aller à l'école , il est resté à la maison .
100.0 is there something in particular that you want to study ? -> y a - t - il quelque chose en particulier que tu veuilles étudier ?
100.0 i have as many books as he does . -> j'ai autant de livres que lui .
100.0 i studied french for three hours last night . -> j'ai étudié le français pendant trois heures la nuit dernière .
100.0 tom seems to be aware of what's happening . -> tom semble être conscient de ce qui se passe .
Bahdanau (concat) : elle lui a conseillé de parler de sa vie en amérique .


Luong dot : elle lui a conseillé de parler de sa vie en amérique .
Luong general : elle lui conseilla de parler en vie de sa vie .
